In [1]:
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig

In [2]:
URL = 'https://luma.com/la'

import re
import asyncio
from crawl4ai import AsyncWebCrawler


async def main():
    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(
            url=URL,
        )
        return result

import asyncio
# asyncio.run(crawl())   
result = await main()
result

[INIT].... → Crawl4AI 0.9.2 

[FETCH]... ↓ https://luma.com/la                                                                                  | ✓ | ⏱: 1.13s 

[SCRAPE].. ◆ https://luma.com/la                                                                                  | ✓ | ⏱: 0.03s 

[COMPLETE] ● https://luma.com/la                                                                                  | ✓ | ⏱: 1.17s 

CrawlResultContainer([CrawlResult(url='https://luma.com/la', html='<!DOCTYPE html><html class="theme-root light" lang="en"><head><meta charset="utf-8" data-next-head=""><meta name="viewport" content="width=device-width,height=device-height,initial-scale=1,maximum-scale=1,user-scalable=no,shrink-to-fit=no,viewport-fit=cover" data-next-head=""><link rel="shortcut icon" href="/favicon.ico" data-next-head=""><link rel="apple-touch-icon" href="/apple-touch-icon.png" data-next-head=""><title data-next-head="">Popular events in Los Angeles · Events Calendar</title><meta name="description" content="Discover the hottest events in Los Angeles, and get notified of new events before they sell out." data-next-head=""><meta property="og:title" content="Popular events in Los Angeles · Events Calendar" data-next-head=""><meta property="og:description" content="Discover the hottest events in Los Angeles, and get notified of new events before they sell out." data-next-head=""><meta property="og:type" co

In [3]:
result.markdown

"[Skip to Content](https://luma.com/la#main-content)[](https://luma.com/)\n5:12 PM PDT\n[Discover Events](https://luma.com/discover)\n[Sign In](https://luma.com/signin?next=%2Fla)\n![](https://images.lumacdn.com/discovery/la-icon.png)\n## What’s Happening in\n# Los Angeles\n5:12 PM PDT\n* * *\nLos Angeles brims with activities for all interests, from wellness experiences to culinary classes and film celebrations. There's always something to discover, learn, and enjoy in this dynamic urban landscape.\n\u200b\nSubscribe\n[](https://luma.com/la/map)\n![Discovery Icon](https://images.lumacdn.com/cdn-cgi/image/format=auto,fit=cover,dpr=2,anim=false,background=%23538f21,quality=75,width=30,height=30/discovery/la-icon.png)\nLos Angeles\nSubscribe\n![](https://images.lumacdn.com/discovery/la-icon.png)\nLos Angeles\nDiscover the hottest events in Los Angeles, and get notified of new events before they sell out.\n\u200b\nSubscribe\n## Events\n[Submit Event](https://luma.com/signin?next=%2Fla)\nT

In [4]:
from crawl4ai import (
    AsyncWebCrawler,
    CrawlerRunConfig,
    LLMConfig, 
    LLMExtractionStrategy, 
    CacheMode
)

from pydantic import BaseModel, Field
import time
import asyncio
from datetime import datetime
# from utils.database import RawEvent

class Event(BaseModel):
    title: str
    start_time: datetime
    location: str
    link: str
    image_link: str

class EventList(BaseModel):
    events: list[Event]

INSTRUCTIONS = (
    "Extract events from the page. "
    "Extract the event title, start date/time, location, "
    "canonical event URL, and image URL. "
    "Return absolute URLs for links and images. "
    "Use the event's own link, not unrelated page links. "
    "If a field is unavailable, do not invent it."
)

strategy = LLMExtractionStrategy(
    llm_config=LLMConfig(
        provider="ollama/dolphin3:latest",
    ),
    schema=EventList.model_json_schema(),
    extraction_type="schema",
    instruction=INSTRUCTIONS,
    # input_format="markdown",
    apply_chunking=False,
    extra_args={"temperature": 0},
    verbose=True,
)

async def main(url: str):
    config = CrawlerRunConfig(
        cache_mode=CacheMode.BYPASS,
        word_count_threshold=5,
        page_timeout=80000,
        extraction_strategy=strategy,
    )

    start = time.perf_counter()

    async with AsyncWebCrawler() as crawler:
        print("Starting crawl...")
        crawl_start = time.perf_counter()

        result = await crawler.arun(url=url, config=config)

        print(f"Total elapsed: {time.perf_counter() - start:.1f}s")
        print(f"Result success: {result.success}")

        # if result.extracted_content:
        #     print(result.extracted_content)

        return result.model_dump()

In [11]:
if __name__ == "__main__":
    result = await main(URL)

[INIT].... → Crawl4AI 0.9.2 

Starting crawl...


[FETCH]... ↓ https://luma.com/la                                                                                  | ✓ | ⏱: 1.16s 

[SCRAPE].. ◆ https://luma.com/la                                                                                  | ✓ | ⏱: 0.03s 

[LOG] Call LLM for https://luma.com/la - block index: 0
[LOG] Extracted 8 blocks from URL: https://luma.com/la block index: 0


[EXTRACT]. ■ https://luma.com/la                                                                                  | ✓ | ⏱: 19.71s 

[COMPLETE] ● https://luma.com/la                                                                                  | ✓ | ⏱: 20.92s 

Total elapsed: 21.2s
Result success: True


/tmp/ipykernel_11289/2694895507.py:2: RuntimeWarning: coroutine 'main' was never awaited
  result = await main(URL)


In [16]:
list(result.keys())

['url',
 'html',
 'fit_html',
 'success',
 'cleaned_html',
 'media',
 'links',
 'downloaded_files',
 'js_execution_result',
 'screenshot',
 'pdf',
 'mhtml',
 'extracted_content',
 'metadata',
 'error_message',
 'session_id',
 'response_headers',
 'status_code',
 'ssl_certificate',
 'dispatch_result',
 'redirected_url',
 'redirected_status_code',
 'network_requests',
 'console_messages',
 'tables',
 'head_fingerprint',
 'cached_at',
 'cache_status',
 'crawl_stats',
 'markdown']

In [ ]:
import json
json.loads(result['extracted_content'])

[{'title': 'Sunday Sprint (Create the best animated pilot in 8 hrs)',
  'start_time': 'Live 10:00 AM',
  'location': '9430 Washington Blvd',
  'link': 'https://luma.com/hzqr6ea4',
  'image_link': 'https://images.lumacdn.com/cdn-cgi/image/format=auto,fit=cover,dpr=2,anim=false,background=white,quality=75,width=180,height=180/uploads/25/9bb98d06-2c47-4268-991a-28688a4be428.gif',
  'error': False},
 {'title': 'BUBBELA X NJR X MAMAN RUN & WELLNESS CLUB',
  'start_time': 'Live 4:00 PM',
  'location': 'Venice',
  'link': 'https://luma.com/8tgne71s',
  'image_link': 'https://images.lumacdn.com/cdn-cgi/image/format=auto,fit=cover,dpr=2,anim=false,background=white,quality=75,width=180,height=180/uploads/f0/c4eec416-5463-4545-9c15-c99910259c3c.png',
  'error': False},
 {'title': 'Los Angeles | Claude for Founders & Fractionals',
  'start_time': '5:00 PM',
  'location': 'The Reserve +99',
  'link': 'https://luma.com/claude-la-fractionals',
  'image_link': 'https://images.lumacdn.com/cdn-cgi/image